# Classifier-Based Evaluation of Generated Images

Train a HybridCNNTransformer on **real** training data, then evaluate on:
- **Real eval images** (TRTR baseline)
- **Generated images** (TRTS — Train Real, Test Synthetic)

If generated images are realistic, a classifier trained on real data should classify them with similar accuracy.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader
from pathlib import Path

from src.dataset import HyperspectralDataset
from src.models.classifier import HybridCNNTransformer
from src.evaluate_classifier import (
    GeneratedDataset, train_one_epoch, evaluate_split,
    per_class_accuracy, confusion_matrix,
)

DATA_DIR = Path("../data/processed")
GEN_DIR = Path("../results/eval/generated")
OUT_DIR = Path("../results/eval")
FIGURES_DIR = Path("../results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Datasets

In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 0

train_ds = HyperspectralDataset(
    split_file=str(DATA_DIR / "train.txt"),
    stats_file=str(DATA_DIR / "stats.npz"),
)
eval_ds = HyperspectralDataset(
    split_file=str(DATA_DIR / "eval.txt"),
    stats_file=str(DATA_DIR / "stats.npz"),
)
gen_ds = GeneratedDataset(str(GEN_DIR), str(DATA_DIR / "stats.npz"))

print(f"Train: {len(train_ds)}, Real Eval: {len(eval_ds)}, Generated: {len(gen_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
gen_loader = DataLoader(gen_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")

## 2. Train Classifier on Real Data

In [ ]:
EPOCHS = 20
LR = 3e-4

# Set to a checkpoint path to skip training, e.g.:
# CLASSIFIER_CKPT = "../results/classifier/best.pt"
CLASSIFIER_CKPT = None

model = HybridCNNTransformer().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

if CLASSIFIER_CKPT:
    ckpt = torch.load(CLASSIFIER_CKPT, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded classifier from {CLASSIFIER_CKPT}")
else:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = GradScaler("cuda") if device.type == "cuda" else None

    history = []
    for epoch in range(1, EPOCHS + 1):
        loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
        history.append({"epoch": epoch, "loss": loss, "acc": acc})
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>2}/{EPOCHS}: loss={loss:.4f}, acc={acc:.1f}%")

    print("Training complete.")

In [ ]:
# Plot training curves (skip if loaded from checkpoint)
if not CLASSIFIER_CKPT:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = [h["epoch"] for h in history]
    ax1.plot(epochs, [h["loss"] for h in history], "b-")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, [h["acc"] for h in history], "g-")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.set_title("Training Accuracy")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "classifier_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

## 3. Evaluate: Real Eval (TRTR) vs Generated (TRTS)

In [ ]:
print("Evaluating on real eval images (TRTR)...")
real_acc, real_preds, real_labels = evaluate_split(model, eval_loader, device)
print(f"  Real Eval Accuracy: {real_acc:.1f}%")

print("Evaluating on generated images (TRTS)...")
gen_acc, gen_preds, gen_labels = evaluate_split(model, gen_loader, device)
print(f"  Generated Accuracy: {gen_acc:.1f}%")

real_pcls = per_class_accuracy(real_preds, real_labels)
gen_pcls = per_class_accuracy(gen_preds, gen_labels)

# Summary table
rows = [{"class": c, "real_acc": real_pcls[c], "gen_acc": gen_pcls[c]} for c in range(10)]
df = pd.DataFrame(rows)
df.loc[len(df)] = {"class": "Mean", "real_acc": real_acc, "gen_acc": gen_acc}
print("\n" + df.to_string(index=False))

## 4. Per-Class Accuracy Comparison

In [ ]:
classes = np.arange(10)
real_accs = [real_pcls[c] for c in classes]
gen_accs = [gen_pcls[c] for c in classes]
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(classes - width/2, real_accs, width, label="Real Eval (TRTR)",
               color="steelblue", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(classes + width/2, gen_accs, width, label="Generated (TRTS)",
               color="coral", edgecolor="black", linewidth=0.5)

ax.axhline(real_acc, color="steelblue", linestyle="--", alpha=0.6, label=f"Real Mean = {real_acc:.1f}%")
ax.axhline(gen_acc, color="coral", linestyle="--", alpha=0.6, label=f"Gen Mean = {gen_acc:.1f}%")

ax.set_xlabel("Disease Severity Class")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Classifier Accuracy: Real vs Generated Images")
ax.set_xticks(classes)
ax.set_ylim(0, 105)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "classifier_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Confusion Matrices

In [ ]:
real_cm = confusion_matrix(real_preds, real_labels)
gen_cm = confusion_matrix(gen_preds, gen_labels)

# Normalize by row (true label) for display
real_cm_norm = real_cm.astype(float) / (real_cm.sum(axis=1, keepdims=True) + 1e-8)
gen_cm_norm = gen_cm.astype(float) / (gen_cm.sum(axis=1, keepdims=True) + 1e-8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, cm_norm, cm_raw, title in [
    (ax1, real_cm_norm, real_cm, f"Real Eval (TRTR) — {real_acc:.1f}%"),
    (ax2, gen_cm_norm, gen_cm, f"Generated (TRTS) — {gen_acc:.1f}%"),
]:
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="equal")
    for i in range(10):
        for j in range(10):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm_raw[i, j]}", ha="center", va="center",
                    fontsize=8, color=color)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))

fig.colorbar(im, ax=[ax1, ax2], shrink=0.8, label="Recall")
plt.suptitle("Confusion Matrices (normalized by true label)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "classifier_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Accuracy Gap Analysis

In [ ]:
gaps = [real_pcls[c] - gen_pcls[c] for c in range(10)]
colors = ["green" if g <= 0 else "red" for g in gaps]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(classes, gaps, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Disease Severity Class")
ax.set_ylabel("Accuracy Gap (Real - Generated, %)")
ax.set_title("Per-Class Accuracy Gap\n(positive = generated harder, negative = generated easier)")
ax.set_xticks(classes)
ax.grid(axis="y", alpha=0.3)

mean_gap = np.mean(gaps)
ax.axhline(mean_gap, color="orange", linestyle="--", label=f"Mean gap = {mean_gap:.1f}%")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "classifier_accuracy_gap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean accuracy gap: {mean_gap:.1f}%")
print(f"Overall accuracy drop: {real_acc:.1f}% → {gen_acc:.1f}% ({real_acc - gen_acc:+.1f}%)")

## 7. Save Results

In [ ]:
# Save CSV
csv_path = OUT_DIR / "classifier_eval.csv"
with open(csv_path, "w") as f:
    f.write("class,real_accuracy,generated_accuracy\n")
    for c in range(10):
        f.write(f"{c},{real_pcls[c]:.2f},{gen_pcls[c]:.2f}\n")
print(f"Saved to {csv_path}")

# Save confusion matrices
np.savez(OUT_DIR / "confusion_matrices.npz",
         real_cm=real_cm, gen_cm=gen_cm,
         real_acc=real_acc, gen_acc=gen_acc)
print(f"Saved confusion matrices to {OUT_DIR / 'confusion_matrices.npz'}")

print(f"\n--- Summary ---")
print(f"Real Eval Accuracy (TRTR): {real_acc:.1f}%")
print(f"Generated Accuracy (TRTS): {gen_acc:.1f}%")
print(f"Accuracy Gap:              {real_acc - gen_acc:+.1f}%")